In [3]:
from src.core.services.connection.databricks_connector import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-mock-data-generator")
conn.connect()

spark = conn.spark

print(" Connected to Spark!")
print(f"Spark Version: {spark.version}")

2026-10-08 14:33:42.322 | INFO     | src.core.services.connection.databricks_connector:connect:34 - SparkSession ready | app=yandex-mock-data-generator | version=4.2.0 | runtime=serverless-connect


 Connected to Spark!
Spark Version: 4.2.0


In [4]:
from dotenv import load_dotenv
import os
load_dotenv()
catalog = os.environ.get("DATABRICKS_CATALOG")
raw_schema = f"{catalog}.raw_files"
volume_name = "landing"

print(f"🔧 Checking infrastructure for catalog [{catalog}]...")


volume_path = f"/Volumes/{catalog}/raw_files/{volume_name}"
print(f" Volume is ready at: {volume_path}")

🔧 Checking infrastructure for catalog [yandex_go_dev]...
 Volume is ready at: /Volumes/yandex_go_dev/raw_files/landing


In [2]:
from src.core.services.data_generation.driver_generator import DriverGenerator

print("🚘 Generating mock drivers stream data...")

driver_gen = DriverGenerator(
    subfolder="json",
    volume_name="landing",
    pool_size=50
)

driver_gen.generate(spark=spark, row_count=50, with_phone=False)

print(f" Drivers stream files successfully written to: {driver_gen.output_path}")

🚘 Generating mock drivers stream data...
 Drivers stream files successfully written to: /Volumes/yandex_go_dev/raw_files/landing/json


In [3]:
from src.core.services.data_generation.user_generator import UserGenerator

print(" Generating mock user profiles...")

user_gen = UserGenerator(
    subfolder="avro",
    volume_name="landing",
    pool_size=500
)

user_gen.generate(spark=spark, row_count=500)

print(f" Users successfully written to: {user_gen.output_path}")

 Generating mock user profiles...
 Users successfully written to: /Volumes/yandex_go_dev/raw_files/landing/avro


In [4]:
from src.core.services.data_generation.trip_generator import TripGenerator

print("🚕 Generating mock taxi trips...")

trip_gen = TripGenerator(
    subfolder="parquet",
    volume_name="landing",
    user_pool_size=500,
    driver_pool_size=100
)

trip_gen.generate(spark=spark, row_count=2000)

print(f" Taxi trips successfully written to: {trip_gen.output_path}")

🚕 Generating mock taxi trips...
 Taxi trips successfully written to: /Volumes/yandex_go_dev/raw_files/landing/parquet


In [14]:
print(" Verifying files in Unity Catalog Volume:")

# 1 Preview Users (Avro)
print("\n--- Preview Users (Avro) ---")
df_users_check = spark.read.format("avro").load(f"{volume_path}/avro")
print(f"Total user records: {df_users_check.count()}")
df_users_check.show(5, truncate=False)

# 2 Preview Drivers (JSON)
print("\n--- Preview Drivers (JSON) ---")
df_drivers_check = spark.read.json(f"{volume_path}/json")
print(f"Total driver records: {df_drivers_check.count()}")
df_drivers_check.show(5, truncate=False)

# 3 Preview Taxi Trips (Parquet)
print("\n--- Preview Taxi Trips (Parquet) ---")
df_trips_check = spark.read.parquet(f"{volume_path}/parquet")
print(f"Total trip records: {df_trips_check.count()}")
(
    df_trips_check
    .select("id", "user_id", "driver_id", "pickup_datetime", "fare_amount", "total_amount")
    .show(5, truncate=False)
)

 Verifying files in Unity Catalog Volume:

--- Preview Users (Avro) ---
Total user records: 500
+------------------------------------+--------------------+-----------------------+-----------------+------------------+
|id                                  |full_name           |email                  |registration_date|is_plus_subscriber|
+------------------------------------+--------------------+-----------------------+-----------------+------------------+
|bdd640fb-0667-4ad1-9c80-317fa3b1799d|Wesley Miller       |wsmith@example.net     |2024-08-13       |false             |
|23b8c1e9-3924-46de-beb1-3b9046685257|Ashley Olsen        |eboyer@example.net     |2020-02-10       |true              |
|bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9|Karen Washington    |dgutierrez@example.com |2020-11-01       |false             |
|972a8469-1641-4f82-8b9d-2434e465e150|Christopher Williams|wclark@example.net     |2022-08-14       |true              |
|17fc695a-07a0-4a6e-8822-e8f36c031199|Linda Brewer       

In [ ]:
conn.disconnect()
print("🛑 Spark session closed cleanly.")